In [ ]:
import sys
if 'google.colab' in sys.modules:
    !uv pip install --system -q --pre ngsolve

# double-slit experiment

In [ ]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

everything in nm

In [ ]:
c = 2.99e8*1e9   # nm/s
lam = 200     # wave length
rad = 2000    # domain size
kappa = 2*pi/lam   # wave number

order=5   # f

In [ ]:
domain = MoveTo(0,0).RectangleC(2*rad,2*rad).Face()
pmlreg =  MoveTo(0,0).RectangleC(2.5*rad,2.5*rad).Face() - domain
pmlreg.name="pml"

screen = MoveTo(500,0).RectangleC(20, 5000).Face()

# one single slit
# screen -= MoveTo(500,0).RectangleC(50, 0.2*lam).Face()    

# two slits
screen -= MoveTo(500,400).RectangleC(50, 0.2*lam).Face()
screen -= MoveTo(500,-400).RectangleC(50, 0.2*lam).Face()

shape = Glue([domain, pmlreg]) - screen
mesh = shape.GenerateMesh(maxh=min(lam,rad/20), dim=2)
Draw (shape);

In [ ]:
mesh.SetPML(pml.BrickRadial(mins=(-rad,-rad), maxs=(rad,rad),alpha=0.3j,origin=(0,0)), "pml")

fes = H1(mesh, order=order, complex=True)
u,v = fes.TnT()

# symmetric source 
src = exp(1j*kappa*x)*exp(-( (x+1000)**2+y**2 ) / (0.3*lam)**2)

# directional source
# src = exp(1j*kappa*x)*exp(-( (x+1000)**2+y**2 ) / (1*lam)**2)

Draw (src, mesh); 

use finite elements to solve

In [ ]:
lhs = grad(u)*grad(v)*dx-kappa**2*u*v*dx   # wave equation
if True:
  with TaskManager():
    bf = BilinearForm(lhs).Assemble()
    lf = LinearForm(src*v*dx).Assemble()
    sol = GridFunction(fes)
    sol.vec.data=bf.mat.Inverse(inverse="sparsecholesky")*lf.vec
else:
    # needs developer version:
    sol = Solve (lhs==src*v*dx, solvers.SparseCholesky.Creator())    

In [ ]:
Draw (sol, animate_complex=True, order=order, min=-100, max=100); # min=0, max=1000, 